# Stage 2 Notebook 23 - Exp2R Anchor-conditioned lane queries (DAB-DETR)

**Why this exists.** Exp2P (NB21) showed query-style cls works on the first try (val_lane_f1=0.65), but geometry collapsed (matched_iou=0.13) because random-init queries take many epochs to specialize spatially in 10 epochs of training.

Exp2R applies the **DAB-DETR** convergence-acceleration pattern from Liu et al. 2022: each of K=12 queries has a learnable `(start_y, start_x, theta)` anchor whose sinusoidal embedding becomes the query's positional encoding. This injects explicit positional bias from initialization, so queries specialize spatially much faster.

Output param head produces a **delta** added to the anchor (gentle refinement, scaled by 0.1), not the absolute curve params from scratch. This is essentially the iterative refinement pattern from DAB-DETR.

DN-DETR scaffolding is also added in `LaneQueryHeadAnchorDN` (denoising queries from noised GT) but is **disabled by default** in this experiment (`dn_num_groups: 0`) because plumbing target tensors through `FusionModel.forward` is a separate change. With DN off, the head reduces to a clean DAB-style anchor-conditioned query head.

Hypothesis: anchors give queries the spatial inductive bias they were missing. Geometry should converge much faster than Exp2P; cls should match or exceed Exp2P (still K=12 binary task with Hungarian matching).

Reference: Liu et al. 2022 'DAB-DETR: Dynamic Anchor Boxes are Better Queries for DETR'; Li et al. 2022 'DN-DETR: Accelerate DETR Training by Introducing Query DeNoising'.

### Run mode

1. Keep `DEBUG_MODE = True` for the first run.
2. After smoke + debug pass, change to `False` for the 10-epoch short run.
3. Output mirrored to notebook cell, Colab runtime log, Drive log file.
4. Do not rerun NB00.

In [1]:
import os, sys, subprocess, textwrap
from google.colab import drive
os.environ['PYTHONUNBUFFERED'] = '1'
drive.mount('/content/drive')

REPO_ROOT = '/content/drive/MyDrive/EcoCAR/yolop_vehicle_lane'
if not os.path.isdir(REPO_ROOT):
    raise FileNotFoundError(f'Missing project root: {REPO_ROOT}')
os.chdir(REPO_ROOT)
if REPO_ROOT not in sys.path:
    sys.path.insert(0, REPO_ROOT)

subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', 'pyyaml', 'scipy', 'opencv-python-headless', 'tqdm', 'matplotlib'])
print('repo:', REPO_ROOT)

from stage2.scripts.notebook_utils import run_streaming
LOG_DIR = '/content/drive/MyDrive/EcoCAR/training_runs/notebook_logs'
os.makedirs(LOG_DIR, exist_ok=True)

Mounted at /content/drive
repo: /content/drive/MyDrive/EcoCAR/yolop_vehicle_lane


In [2]:
from pathlib import Path
import os, sys

CONFIG = 'stage2/configs/exp18_rmt_gca_query_anchor_dab_joint.yaml'
LOG_FILE = os.path.join(LOG_DIR, f'{Path(CONFIG).stem}_smoke.log')
run_streaming([sys.executable, '-u', 'stage2/scripts/smoke_test_joint_models.py', CONFIG], log_path=LOG_FILE)

[run_streaming] command: /usr/bin/python3 -u stage2/scripts/smoke_test_joint_models.py stage2/configs/exp18_rmt_gca_query_anchor_dab_joint.yaml
[run_streaming] log file: /content/drive/MyDrive/EcoCAR/training_runs/notebook_logs/exp18_rmt_gca_query_anchor_dab_joint_smoke.log
OK exp18_rmt_gca_query_anchor_dab_joint.yaml
  lane_shape=(1, 12, 72, 2) det_shape=(1, 4, 4)
  lane_loss=2.4523 det_loss=3.1008 grad_cos=-0.0440 lambda_lane=0.1668
  gate_stats={'gate/det_mean': 0.5012335181236267, 'gate/lane_mean': 0.4989289343357086, 'gate/det_sat_low': 0.0, 'gate/det_sat_high': 0.0, 'gate/lane_sat_low': 0.0, 'gate/lane_sat_high': 0.0}
[run_streaming] return_code=0


0

In [3]:
from pathlib import Path
import os, sys

CONFIG = 'stage2/configs/exp18_rmt_gca_query_anchor_dab_joint.yaml'
CURVE_TAR = '/content/drive/MyDrive/EcoCAR/datasets/bdd100k_clrkd_curve.tar'
CURVE_ROOT = '/content/bdd100k_clrkd_curve'

DEBUG_MODE = False

if DEBUG_MODE:
    RUN_TAG = 'debug'
    EPOCHS = 2
    BATCH_SIZE = 4
    LIMIT_TRAIN = 512
    LIMIT_VAL = 256
    PRINT_EVERY = 5
else:
    RUN_TAG = 'short10'
    EPOCHS = 10
    BATCH_SIZE = 8
    LIMIT_TRAIN = 3000
    LIMIT_VAL = 1000
    PRINT_EVERY = 5

run_stem = Path(CONFIG).stem + '_' + RUN_TAG
WORK_DIR = f'/content/{run_stem}'
OUTPUT_TAR = f'/content/drive/MyDrive/EcoCAR/training_runs/{run_stem}.tar'
LOG_FILE = os.path.join(LOG_DIR, f'{run_stem}_train.log')

cmd = [
    sys.executable, '-u', 'stage2/scripts/train_joint_model_experiment.py',
    '--config', CONFIG,
    '--curve-tar', CURVE_TAR,
    '--curve-root', CURVE_ROOT,
    '--work-dir', WORK_DIR,
    '--output-tar', OUTPUT_TAR,
    '--epochs', str(EPOCHS),
    '--batch-size', str(BATCH_SIZE),
    '--limit-train', str(LIMIT_TRAIN),
    '--limit-val', str(LIMIT_VAL),
    '--force-extract',
    '--print-every', str(PRINT_EVERY),
]

print('DEBUG_MODE:', DEBUG_MODE, flush=True)
print('About to run:', ' '.join(cmd), flush=True)
print('Output tar:', OUTPUT_TAR, flush=True)
print('Visible log file:', LOG_FILE, flush=True)
run_streaming(cmd, log_path=LOG_FILE)

DEBUG_MODE: False
About to run: /usr/bin/python3 -u stage2/scripts/train_joint_model_experiment.py --config stage2/configs/exp18_rmt_gca_query_anchor_dab_joint.yaml --curve-tar /content/drive/MyDrive/EcoCAR/datasets/bdd100k_clrkd_curve.tar --curve-root /content/bdd100k_clrkd_curve --work-dir /content/exp18_rmt_gca_query_anchor_dab_joint_short10 --output-tar /content/drive/MyDrive/EcoCAR/training_runs/exp18_rmt_gca_query_anchor_dab_joint_short10.tar --epochs 10 --batch-size 8 --limit-train 3000 --limit-val 1000 --force-extract --print-every 5
Output tar: /content/drive/MyDrive/EcoCAR/training_runs/exp18_rmt_gca_query_anchor_dab_joint_short10.tar
Visible log file: /content/drive/MyDrive/EcoCAR/training_runs/notebook_logs/exp18_rmt_gca_query_anchor_dab_joint_short10_train.log
[run_streaming] command: /usr/bin/python3 -u stage2/scripts/train_joint_model_experiment.py --config stage2/configs/exp18_rmt_gca_query_anchor_dab_joint.yaml --curve-tar /content/drive/MyDrive/EcoCAR/datasets/bdd100k

0

## What to watch in Exp2R training

Reference recent results:
- Exp2P (queries, no anchor): matched_iou=0.13, val_lane_f1=0.65, decoded_f1=0.026, oracle_f1=0.07.
- Exp2N (priors): matched_iou=0.42, val_lane_f1=0.05, decoded_f1=0.011, oracle_f1=0.27.

Pass criteria at epoch 10:

- **`val/matched_line_iou >= 0.25`**: anchors should accelerate geometry convergence vs Exp2P's 0.13. Closer to prior-based 0.42 is the goal.
- **`val/lane_exist_best_f1 >= 0.55`**: cls should hold near Exp2P's 0.67 (anchors don't change the cls task).
- **`val/lane/decoded_f1 >= 0.08`**: combining both improvements should beat Exp2P's 0.026 by at least 3x.
- **`val/lane/decoded_oracle_f1 >= 0.15`**: better geometry should lift the oracle ceiling.

Failure signals -> next ablation:

- Geometry still collapses (matched_iou < 0.20): anchors not enough; need actual denoising queries (turn on `dn_num_groups: 4` once FusionModel can pass targets through).
- Cls regresses (best_f1 < 0.40): the anchor delta + sigmoid scheme may be over-constraining the predictions. Loosen by removing the 0.1 multiplier on param_delta.